- **I ('T_dim', 'X_dim', 'Color_dim') (1200, 1181, 3)**:    8-bit image color values of the timestack. Three dimensions: time, spatial axis, color band, where the colors are RGB (R=0, G=1, B=2). The horizontal axis of the image is the spatial axis. The different crs mappings represent the same coordinates in UTM, local, and longitude/latitude.
- **crs_utm () ()**:  Coordinate reference system definition for UTM coordinates.
- **crs_latlon () ()**:    Coordinate reference system definition for longitude/latitude coordinates.
- **Ri ('T_dim',) (1200,)**:    Pixel coordinate of wave runup line as found by the SegFormer model.
- **runup_val () ()**:    Runup value used to extract runup from softmax score.
- **rundown_val () ()**:    Rundown value used to extract rundown from softmax score.
- **U_transect ('X_dim',) (1181,)**:    Pixel coordinate along the horizontal axis of the image where the timestack was sampled.
- **V_transect ('X_dim',) (1181,)**:    Pixel coordinate along the vertical axis of the image where the timestack was sampled.
- **T ('T_dim',) (1200,)**:    Times that pixels were sampled to create the timestack. Format: YYYY-MM-DD HH:mm:SS+00:00. The dimension length equals the number of samples in the timestack.
- **eastings_transect ('X_dim',) (1181,)**:    Easting coordinates of timestack pixels projected onto the beach surface, described in UTM Easting meters.
- **northings_transect ('X_dim',) (1181,)**:    Northing coordinates of timestack pixels projected onto the beach surface, described in UTM Northing meters.
- **elevation_transect ('X_dim',) (1181,)**:    Elevation (z-value) of timestack pixels projected onto the beach surface.
- **localX_transect ('X_dim',) (1181,)**:    Local cross-shore coordinates in meters of the rectified image. Rotated based on shore-normal angle and origin.
- **localY_transect ('X_dim',) (1181,)**:    Local along-shore coordinates in meters of the rectified image. Rotated based on shore-normal angle and origin
- **Hrunup_utm ('T_dim', 'XY_dim') (1200, 2)**:    Horizontal coordinates of wave runup converted from image coordinates (U,V) to Eastings and Northings.
- **Hrunup_local ('T_dim', 'XY_dim') (1200, 2)**:    Horizontal coordinates of wave runup in the local coordinate system. Rotated based on shore-normal angle and origin.
- **TWL ('T_dim',) (1200,)**:    Vertical elevation of total water level.
- **Zrunup ('T_dim',) (1200,)**:    Vertical elevation of wave runup (total water level minus TWL_tideWindSetup).
- **TWLstats_2exceedence_peaks () ()**:    2 percent exceedance value for TWL peaks.
- **TWLstats_2exceedence_notpeaks () ()**:    2 percent exceedance value for the full TWL time series.
- **TWLstats_setup () ()**:    Wave setup.
- **TWLstats_Tpeak () ()**:    Runup (or TWL) time series peak period.
- **TWLstats_Tmean () ()**:    Runup (or TWL) time series mean period.
- **TWLstats_Ssig () ()**:    Significant swash.
- **TWLstats_Ssig_SS () ()**:    Significant swash in the sea-swell band.
- **TWLstats_Ssig_IG () ()**:    Significant swash in the infragravity band.
- **TWLstats_spectrum ('TWLstats_dim',) (151,)**:    Runup (or TWL) spectral density array.
- **TWLstats_frequency ('TWLstats_dim',) (151,)**:    Runup (or TWL) frequency array.
- **TWLstats_betaS2006 () ()**:    Beach slope between ±2 standard deviations of the TWL time series, as defined in Stockdon et al. (2006).
- **TWLstats_beta () ()**:    Beach slope between +max(TWL) and -min(TWL) elevations.
- **twl () ()**:    TWL forecast.
- **twl05 () ()**:    TWL forecast 5% confidence interval.
- **twl95 () ()**:    TWL forecast 95% confidence interval.
- **twl_setup () ()**:    TWL forecast setup component.
- **twl_runup () ()**:    TWL forecast runup component.
- **twl_runup05 () ()**:    TWL forecast runup 5% confidence interval.
- **twl_runup95 () ()**:    TWL forecast runup 95% confidence interval.
- **twl_tide () ()**:    TWL forecast tide + wind setup component.
- **twl_swash () ()**:    TWL forecast swash component.
- **twl_incSwash () ()**:    TWL forecast incident-band swash component.
- **twl_igSwash () ()**:    TWL forecast infragravity-band swash component.
- **twl_hs () ()**:    TWL forecast offshore significant wave height (Hs).
- **twl_pp () ()**:    TWL forecast peak wave period (Tp).
- **X_dim ('X_dim',) (1181,)**:    Spatial dimension index of the timestack transect.
- **T_dim ('T_dim',) (1200,)**:    Temporal dimension index of the timestack.
- **Color_dim ('Color_dim',) (3,)**: RGB color band dimension (0=Red, 1=Green, 2=Blue).

Plots a time series of water levels for a specified burst 

In [ ]:
%matplotlib inline

import netCDF4 as nc
from datetime import datetime, timezone
import numpy as np
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

nc_dir = Path("E:/cvolpano/shorescan_processing/results/netcdf")
output_dir = nc_dir.parent / "output_plots"

# Regex to robustly parse filenames like:
# 1728990000.Tue.Oct.15_11_00_00.GMT.2024.CACO03.c1.ras_transect20.nc
FNAME_RE = re.compile(
    r"^(?P<timestamp>\d+)\."
    r"(?P<dow>\w+)\."
    r"(?P<month>\w+)\."
    r"(?P<day>\d+)_\d+_\d+_\d+\."
    r"GMT\."
    r"(?P<year>\d+)\."
    r"(?P<camera>\w+)\."
    r"c1\."
)

fig, ax = plt.subplots(figsize=(12, 4))
plt.ylabel('Projected Elevation (m)')
plt.xlabel('Cross shore coordinates (m)')
plt.grid(True)

transects = [20, 30, 40, 50, 70, 100, 150]

cmap = plt.colormaps['Spectral']
colors = [cmap(i / (len(transects) - 1)) for i in range(len(transects))]

# Metadata captured from the first successfully parsed file, used for title/filename
meta = None

for i, transect in enumerate(transects):
    file_path = nc_dir / (
        f"1728990000.Tue.Oct.15_11_00_00.GMT.2024.CACO03.c1."
        f"ras_transect{transect}.nc"
    )
    print(file_path)

    if not file_path.exists():
        print(f"Skipping file {file_path}, path does not exist")
        continue

    fname = file_path.name
    m = FNAME_RE.match(fname)
    if not m:
        print(f"Skipping file {file_path}, filename did not match expected pattern")
        continue

    timestamp = int(m.group("timestamp"))
    month = m.group("month")
    day = m.group("day")
    year = m.group("year")
    camera = m.group("camera")

    dt0 = datetime.fromtimestamp(timestamp, tz=timezone.utc)

    try:
        with xr.open_dataset(file_path) as ds:
            x = np.asarray(ds['localX_transect'])
            elev = np.asarray(ds['elevation_transect'])
    except Exception as e:
        print(f"Failed to process {file_path}: {e}")
        continue

    plt.plot(x, elev, label=f'Transect {transect}', c=colors[i])

    if meta is None:
        meta = {"month": month, "day": day, "year": year, "camera": camera}

if ax.has_data():
    plt.legend(loc='upper right')
    plt.xlim([0, 150])
    plt.ylim([-1, 7])
    plt.title(
        f"Shorescan Transect Elevations ({meta['camera']})\n"
        f"{meta['month']} {meta['day']}, {meta['year']}"
    )
    plt.grid(True)

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / (
        f"Transect_elevations_{meta['camera']}_{meta['month']}_"
        f"{meta['day']}_{meta['year']}.png"
    )
    plt.savefig(out_path)
    print(f"Plot saved to {out_path}")
    plt.show()
else:
    print("No data found, figure not saved")

Process a directory of netcdfs representing burst samples at multiple transects, group by transect, calculate statistics, and prepare for plotting.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import netCDF4 as nc
from datetime import datetime, timezone
from collections import defaultdict
from pathlib import Path
import xarray as xr

# Folder containing runup NetCDFs
nc_dir = Path("E:/cvolpano/shorescan_processing/results/netcdf")
file_list = sorted(nc_dir.glob("*.nc"))

# Dictionary grouped by (transect, camera)
records = defaultdict(list)

for file_path in file_list:
    fname = os.path.basename(file_path)

    # Extract metadata from filename
    transect_match = re.search(r'transect(\d+)', fname)
    camera_match = re.search(r'\.(c\d)\.', fname)

    if transect_match is None:
        print(f"Skipping {fname}: no transect found")
        continue

    transect = int(transect_match.group(1))
    camera = camera_match.group(1) if camera_match else "unknown"

    # Extract datetime from unix timestamp (seconds, not ns)
    timestamp = int(fname.split(".", 1)[0])
    dt0 = pd.Timestamp(timestamp, unit="s", tz="UTC").tz_convert("US/Eastern")

    try:
        ds = xr.open_dataset(file_path, chunks={"T": 100})

        def safe_mean(varname):
            """Return nanmean of a variable if it exists, else np.nan."""
            if varname in ds.variables:
                arr = ds[varname].load()  # Load into memory respecting chunks
                return float(np.nanmean(arr))
            return np.nan

        def safe_percentile(varname, pct):
            """Return percentile if variable exists, else np.nan."""
            if varname in ds.variables:
                arr = ds[varname].load()
                return float(np.percentile(arr, pct))
            return np.nan

        # CoastCam values
        cc_twl = safe_mean("twl")
        cc_Hs = safe_mean("twl_hs")
        cc_ru_2pct = safe_percentile("twl_runup", 98)
        cc_tide = safe_mean("twl_tide")
        cc_beta06 = safe_mean("twl_betaS2006")

        # Shorescan values
        ss_twl = ds["TWL"].load() if "TWL" in ds.variables else np.array([])
        ss_zrunup = ds["Zrunup"].load() if "Zrunup" in ds.variables else np.array([])
        ss_beta06 = safe_mean("TWLstats_betaS2006")

        ss_twl_mean = float(np.nanmean(ss_twl)) if ss_twl.size else np.nan
        ss_twl_max = float(np.nanmax(ss_twl)) if ss_twl.size else np.nan
        ss_twl_std = float(np.nanstd(ss_twl)) if ss_twl.size else np.nan
        ss_zrunup_mean = float(np.nanmean(ss_zrunup)) if ss_zrunup.size else np.nan
        ss_zrunup_std = float(np.nanstd(ss_zrunup)) if ss_zrunup.size else np.nan
        ss_ru_2pct = float(np.percentile(ss_zrunup, 98)) if ss_zrunup.size else np.nan

        vals = {
            "dt0": dt0,
            "cc_twl": cc_twl,
            "cc_Hs": cc_Hs,
            "cc_tide": cc_tide,
            "cc_ru_2pct": cc_ru_2pct,
            "ss_ru_2pct": ss_ru_2pct,
            "ss_twl_mean": ss_twl_mean,
            "ss_twl_max": ss_twl_max,
            "ss_twl_std": ss_twl_std,
            "ss_zrunup_mean": ss_zrunup_mean,
            "ss_zrunup_std": ss_zrunup_std,
            "transect": transect,
            "camera": camera,
            "filename": fname,
            "timestamp": timestamp,
            "beta06": ss_beta06,
        }

        # Check that all numeric fields are finite
        if all(
            np.all(np.isfinite(v))
            for v in vals.values()
            if isinstance(v, (np.ndarray, float, int, np.number))
        ):
            records[(transect, camera)].append(vals)
        else:
            print(f"Skipping {fname}: non-finite values")

    except Exception as e:
        print(f"Error processing {fname}: {e}")


# Write one CSV per transect+camera combination, once, after the loop
out_dir = nc_dir  # change if you want a separate output folder
for (transect, camera), rows in records.items():
    df_group = pd.DataFrame(rows).sort_values("dt0")
    out_name = f"Shorescan_TWL_viewer_Transect{transect}_{camera}_sensor_clean.csv"
    df_group.to_csv(out_dir / out_name, index=False)
    print(f"Wrote {len(df_group)} rows to {out_name}")

# Optional: also write one combined CSV per transect (across cameras)
by_transect = defaultdict(list)
for (transect, camera), rows in records.items():
    by_transect[transect].extend(rows)

for transect, rows in by_transect.items():
    df_all = pd.DataFrame(rows).sort_values(["camera", "dt0"])
    out_name = f"Shorescan_TWL_viewer_Transect{transect}_ALLCAMERAS_sensor_clean.csv"
    df_all.to_csv(out_dir / out_name, index=False)
    print(f"Wrote {len(df_all)} rows to {out_name}")

Make a set of plots of water level timeseries for each transect to compare Max Shorescan and TWLCC Viewer

In [ ]:
%matplotlib inline

import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import scipy as sp
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# ==============================================================================
df_ss = pd.read_csv("E:/cvolpano/shorescan_processing/results/netcdf/Shorescan_TWL_viewer_Transect20_ALLCAMERAS_sensor_clean.csv")
df_ss = df_ss.reset_index()

# Convert timestamp FIRST
df_ss["timestamp"] = pd.to_datetime(df_ss["timestamp"], errors="coerce")
df_ss["timestamp"] = df_ss["timestamp"].dt.as_unit("s")

# THEN derive date from the now-datetime column
df_ss["date"] = df_ss["timestamp"].dt.date
# ==============================================================================
df_thresh = pd.read_csv(
    "C:/Users/cvolpano/shorescan_processing/cumulative_power_thresholds_2024-10-01-2025-03-31.csv"
)
df_thresh["timestamp"] = pd.to_datetime(df_thresh["datetime_crossed"], errors="coerce")
df_thresh["timestamp"] = df_thresh["timestamp"].dt.as_unit("s")
# ==============================================================================
df_ss = df_ss.sort_values("timestamp")
df_thresh = df_thresh.sort_values("timestamp")

# For each wave-data row, find the single nearest threshold-crossing event in time,
# searching both backward and forward, and picking whichever is closer.

merge_back = pd.merge_asof(
    df_ss, df_thresh, on="timestamp", direction="backward",
    suffixes=("", "_thresh")
)
merge_fwd = pd.merge_asof(
    df_ss, df_thresh, on="timestamp", direction="forward",
    suffixes=("", "_thresh")
)

# Distance (in time) from each row's timestamp to the matched threshold-crossing timestamp
# The matched crossing datetime is in df_thresh's own 'timestamp' col, carried over via suffix
back_dt = merge_back["timestamp_thresh"] if "timestamp_thresh" in merge_back.columns else merge_back["datetime_crossed"]
fwd_dt = merge_fwd["timestamp_thresh"] if "timestamp_thresh" in merge_fwd.columns else merge_fwd["datetime_crossed"]

back_delta = (df_ss["timestamp"].values - pd.to_datetime(back_dt).values)
fwd_delta = (pd.to_datetime(fwd_dt).values - df_ss["timestamp"].values)

back_delta = pd.to_timedelta(back_delta)
fwd_delta = pd.to_timedelta(fwd_delta)

# Pick whichever match is closer in time (NaT counts as "infinitely far")
use_forward = fwd_delta.fillna(pd.Timedelta.max) < back_delta.fillna(pd.Timedelta.max)

df_combined = merge_back.copy()
df_combined.loc[use_forward, merge_fwd.columns] = merge_fwd.loc[use_forward, merge_fwd.columns].values

# ==============================================================================
# 4. ROBUST SCALAR FLATTENING
# ==============================================================================
def clean_array_element(val):
    if isinstance(val, (list, np.ndarray)):
        try:
            return np.asarray(val).item() if np.asarray(val).size > 0 else np.nan
        except ValueError:
            return np.asarray(val).flat[0]
    return val

vectorized_cleaner = np.vectorize(clean_array_element, otypes=[object])

df_combined["cc_twl"] = vectorized_cleaner(df_combined["cc_twl"])
df_combined["ss_twl_mean"] = vectorized_cleaner(df_combined["ss_twl_mean"])
df_combined["threshold_kWh_m"] = vectorized_cleaner(df_combined["threshold_kWh_m"])

df_combined["cc_twl"] = pd.to_numeric(df_combined["cc_twl"], errors="coerce")
df_combined["ss_twl_mean"] = pd.to_numeric(df_combined["ss_twl_mean"], errors="coerce")
df_combined["threshold_kWh_m"] = pd.to_numeric(df_combined["threshold_kWh_m"], errors="coerce")

# Round to avoid float mismatches like 199.9999998 vs 200
df_combined["threshold_kWh_m"] = df_combined["threshold_kWh_m"].round(0)

df_combined = df_combined.dropna(subset=["cc_twl", "ss_twl_mean", "threshold_kWh_m"])

threshold_palette = {200: "skyblue", 400: "gold", 600: "red"}
df_200 = df_combined[df_combined['threshold_kWh_m']<=200]
df_200['threshold_kWh_m'] = 200

df_400 = df_combined[df_combined['threshold_kWh_m']<=400]
df_400['threshold_kWh_m'] = 400

df_600 = df_combined[df_combined['threshold_kWh_m']<=600]
df_600['threshold_kWh_m'] = 600



df_combined_cat = pd.concat([df_200,df_400,df_600], axis = 0)

present_thresholds = sorted(df_combined_cat["threshold_kWh_m"].unique(),reverse=True)
print(present_thresholds)
palette = [threshold_palette[t] for t in present_thresholds if t in threshold_palette]

g = sns.lmplot(
    data=df_combined_cat,
    x="cc_twl",
    y="ss_twl_mean",
    hue="threshold_kWh_m",
    hue_order=present_thresholds,
    palette=palette,
    height=8,
    legend=False,
    scatter_kws = {
        "linewidths": 0.3,
        "s": 15,
        "alpha": 0.8,
        "edgecolor": 'k',
    }
)

ax = g.ax
limits = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1]),
]
ax.plot(limits, limits, color="black", linestyle="-", label="1:1 Line")

# ax.set_xlim([-2.5, 2.5])
# ax.set_ylim([-2.5, 2.5])
ax.set_title(f"Transect {df_ss['transect'][0]} \n {min(df_combined['timestamp'])} - {max(df_combined['timestamp'])}")
ax.set_xlabel("Coastal Change Total Water Level")
ax.set_ylabel("Shorescan Mean Total Water Level")
ax.grid(True, linestyle="--", alpha=0.5)

r2_200, p = sp.stats.pearsonr(df_200['cc_twl'], df_200['ss_twl_mean'])
r2_400, p = sp.stats.pearsonr(df_400['cc_twl'], df_400['ss_twl_mean'])
r2_600, p = sp.stats.pearsonr(df_600['cc_twl'], df_600['ss_twl_mean'])

    

legend_handles = [
    Line2D([0], [0], marker='o', markerfacecolor=palette[0],  markersize=8, markeredgecolor='black',linewidth=0.3,linestyle='None', label=f"200 kWh/m \n$R^2$: {round(r2_200,2)}"),
    Line2D([0], [0], marker='o', markerfacecolor = palette[1], markersize=8,markeredgecolor='black',linewidth=0.3, linestyle='None', label=f"400 kWh/m \n$R^2$:{round(r2_400,2)}"),
    Line2D([0], [0], marker = 'o', markerfacecolor = palette[2], markeredgecolor='black',linewidth=0.3,markersize = 8, linestyle='None',label=f"600 kWh/m \n$R^2$: {round(r2_600,2)}")
    
]
plt.legend(handles=legend_handles, loc="lower right", frameon=True)
plt.tight_layout()
plt.savefig(f"TWLvSS_{min(df_combined['timestamp'].dt.date)}-{max(df_combined['timestamp'].dt.date)}_Transect{df_ss['transect'][0]}.png")
plt.show()



In [ ]:
# Plot of wave time series with intervals highlighted

In [ ]:
# load wave data

# load thresholds
df_thresh = pd.read_csv(
    "C:/Users/cvolpano/shorescan_processing/cumulative_power_thresholds_2024-10-01-2025-03-31.csv"
)
# plot
